<a href="https://colab.research.google.com/github/icosahedron31/Walmart-Sales/blob/main/model_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mlflow dagshub xgboost lightgbm scikit-learn -q

from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('/content/drive/MyDrive/ColabNotebooks/kaggle_API_credentials/kaggle.json',
            os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import dagshub
import sys
from scipy.optimize import minimize

sys.path.append('/content/drive/MyDrive/walmart/Transformers')
from CategoryConverter import CategoryConverter
from LabelEncoder import LabelEncoderTransformer

dagshub.init(repo_owner='icosahedron31', repo_name='Walmart-Sales', mlflow=True)
mlflow.set_experiment('Model_Inference')

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=c333d51a-ed05-404b-87bf-2c07535ca20e&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c98b70b9f339611d7d1d0de12e2d8db4f57247e21ce8d516889f5adae561d38d




Accessing as njvar23

Initialized MLflow to track repo "icosahedron31/Walmart-Sales"

Repository icosahedron31/Walmart-Sales initialized!

2026/07/12 14:35:18 INFO mlflow.tracking.fluent: Experiment with name 'Model_Inference' does not exist. Creating a new experiment.


In [2]:
train = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/train_processed.csv')
val = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/val_processed.csv')

train['Date'] = pd.to_datetime(train['Date'])
val['Date'] = pd.to_datetime(val['Date'])

train = train.dropna(subset=['lag_52', 'rolling_mean_52'])
val = val.dropna(subset=['lag_52', 'rolling_mean_52'])

print(f"Train: {train.shape}, Val: {val.shape}")

Train: (173973, 29), Val: (87110, 29)


In [3]:
df_sales_train = pd.read_csv('/content/drive/MyDrive/walmart/train.csv')
df_test = pd.read_csv('/content/drive/MyDrive/walmart/test.csv')
df_features = pd.read_csv('/content/drive/MyDrive/walmart/features.csv')
df_stores = pd.read_csv('/content/drive/MyDrive/walmart/stores.csv')

df_sales_train['Date'] = pd.to_datetime(df_sales_train['Date'])
df_test['Date'] = pd.to_datetime(df_test['Date'])
df_features['Date'] = pd.to_datetime(df_features['Date'])

df_features_clean = df_features.drop(columns=['IsHoliday'])

train_full = df_sales_train.merge(df_features_clean, on=["Store", "Date"], how="left")
train_full = train_full.merge(df_stores, on="Store", how="left")

test_full = df_test.merge(df_features_clean, on=["Store", "Date"], how="left")
test_full = test_full.merge(df_stores, on="Store", how="left")
test_full['Weekly_Sales'] = np.nan

train_full = train_full.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_full = test_full.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print(f"Train_full: {train_full.shape}, Test_full: {test_full.shape}")

Train_full: (421570, 16), Test_full: (115064, 16)


In [4]:
full_data = pd.concat([train_full, test_full], ignore_index=True).sort_values(
    ['Store', 'Dept', 'Date']).reset_index(drop=True)

full_data['Week'] = full_data['Date'].dt.isocalendar().week.astype(int)
full_data['Month'] = full_data['Date'].dt.month
full_data['Year'] = full_data['Date'].dt.year
full_data['Quarter'] = full_data['Date'].dt.quarter

superbowl = pd.to_datetime(['2010-02-12','2011-02-11','2012-02-10','2013-02-08'])
thanksgiving = pd.to_datetime(['2010-11-26','2011-11-25','2012-11-23'])
christmas = pd.to_datetime(['2010-12-31','2011-12-30','2012-12-28'])
laborday = pd.to_datetime(['2010-09-10','2011-09-09','2012-09-07'])

full_data['holiday_name'] = 'none'
full_data.loc[full_data['Date'].isin(superbowl), 'holiday_name'] = 'superbowl'
full_data.loc[full_data['Date'].isin(thanksgiving), 'holiday_name'] = 'thanksgiving'
full_data.loc[full_data['Date'].isin(christmas), 'holiday_name'] = 'christmas'
full_data.loc[full_data['Date'].isin(laborday), 'holiday_name'] = 'laborday'

full_data['IsHoliday'] = full_data['IsHoliday'].fillna(False).astype(int)

# MarkDowns: NaN means "no markdown running"
markdown_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
full_data[markdown_cols] = full_data[markdown_cols].fillna(0)

# Economic indicators: forward/back-fill within each store (matches training preprocessing)
for col in ['CPI', 'Unemployment', 'Fuel_Price', 'Temperature']:
    full_data[col] = full_data.groupby('Store')[col].transform(lambda s: s.ffill().bfill())

In [12]:
ensemble_model = mlflow.pyfunc.load_model("models:/Ensemble_Best/1")
wrapped = ensemble_model.unwrap_python_model()

lgbm_model = wrapped.lgbm_model
xgb_model = wrapped.xgb_model
w_lgbm = wrapped.w_lgbm
w_xgb = wrapped.w_xgb
lgbm_expected_order = wrapped.lgbm_feature_order
xgb_expected_order = wrapped.xgb_feature_order

print("Ensemble_Best loaded and unwrapped successfully")

Ensemble_Best loaded and unwrapped successfully


In [13]:
X_val_lgbm = val[lgbm_expected_order].copy()
X_val_xgb = val[xgb_expected_order].copy()
X_val_xgb['IsHoliday'] = X_val_xgb['IsHoliday'].astype(int)

y_val = val['Weekly_Sales']
is_holiday_val = val['IsHoliday']

lgbm_val_preds = lgbm_model.predict(X_val_lgbm)
xgb_val_preds = xgb_model.predict(X_val_xgb)

lgbm_wmae = wmae(y_val, lgbm_val_preds, is_holiday_val)
xgb_wmae = wmae(y_val, xgb_val_preds, is_holiday_val)
print(f"LightGBM Validation WMAE: {lgbm_wmae:.2f}")
print(f"XGBoost Validation WMAE: {xgb_wmae:.2f}")

simple_ensemble_preds = 0.5 * lgbm_val_preds + 0.5 * xgb_val_preds
simple_wmae = wmae(y_val, simple_ensemble_preds, is_holiday_val)
print(f"Simple Ensemble (50/50) WMAE: {simple_wmae:.2f}")

def ensemble_wmae_loss(weights):
    w1, w2 = weights
    preds = w1 * lgbm_val_preds + w2 * xgb_val_preds
    return wmae(y_val, preds, is_holiday_val)

result = minimize(
    ensemble_wmae_loss,
    x0=[0.5, 0.5],
    constraints={'type': 'eq', 'fun': lambda w: sum(w) - 1},
    bounds=[(0, 1), (0, 1)]
)
w_lgbm, w_xgb = result.x
optimized_preds = w_lgbm * lgbm_val_preds + w_xgb * xgb_val_preds
optimized_wmae = wmae(y_val, optimized_preds, is_holiday_val)

print(f"\nOptimized weights: LightGBM={w_lgbm:.3f}, XGBoost={w_xgb:.3f}")
print(f"Optimized Ensemble WMAE: {optimized_wmae:.2f}")

with mlflow.start_run(run_name='Ensemble_Final'):
    mlflow.log_metric('lgbm_val_wmae', lgbm_wmae)
    mlflow.log_metric('xgb_val_wmae', xgb_wmae)
    mlflow.log_metric('simple_ensemble_wmae', simple_wmae)
    mlflow.log_metric('optimized_ensemble_wmae', optimized_wmae)
    mlflow.log_param('lgbm_weight', round(w_lgbm, 3))
    mlflow.log_param('xgb_weight', round(w_xgb, 3))
    print("Ensemble validation results logged to MLflow")


LightGBM Validation WMAE: 1360.32
XGBoost Validation WMAE: 1341.16
Simple Ensemble (50/50) WMAE: 1326.42

Optimized weights: LightGBM=0.399, XGBoost=0.601
Optimized Ensemble WMAE: 1325.44
Ensemble validation results logged to MLflow
🏃 View run Ensemble_Final at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/6/runs/9a4ec79fdf254de88bea725efb469c70
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/6


In [15]:
# Register the ensemble as a single MLflow pyfunc model
import mlflow.pyfunc

class EnsemblePyfunc(mlflow.pyfunc.PythonModel):
    def __init__(self, lgbm_model, xgb_model, w_lgbm, w_xgb, lgbm_feature_order, xgb_feature_order):
        self.lgbm_model = lgbm_model
        self.xgb_model = xgb_model
        self.w_lgbm = w_lgbm
        self.w_xgb = w_xgb
        self.lgbm_feature_order = lgbm_feature_order
        self.xgb_feature_order = xgb_feature_order

    def predict(self, context, model_input):
        # model_input must already contain all raw feature columns
        # (Type/holiday_name as strings — each sub-pipeline encodes internally)
        X_lgbm = model_input[self.lgbm_feature_order].copy()

        X_xgb = model_input[self.xgb_feature_order].copy()
        X_xgb['IsHoliday'] = X_xgb['IsHoliday'].astype(int)

        lgbm_preds = self.lgbm_model.predict(X_lgbm)
        xgb_preds = self.xgb_model.predict(X_xgb)

        combined = self.w_lgbm * lgbm_preds + self.w_xgb * xgb_preds
        return np.clip(combined, 0, None)


with mlflow.start_run(run_name='Ensemble_Pipeline_Final'):
    ensemble_model = EnsemblePyfunc(
        lgbm_model=lgbm_model,
        xgb_model=xgb_model,
        w_lgbm=w_lgbm,
        w_xgb=w_xgb,
        lgbm_feature_order=lgbm_expected_order,
        xgb_feature_order=xgb_expected_order
    )

    mlflow.log_param('lgbm_weight', round(w_lgbm, 3))
    mlflow.log_param('xgb_weight', round(w_xgb, 3))
    mlflow.log_metric('val_wmae', optimized_wmae)
    mlflow.log_metric('kaggle_public_wmae', 2928.32154)
    mlflow.log_metric('kaggle_private_wmae', 3067.90384)

    mlflow.pyfunc.log_model(
        artifact_path='ensemble_pipeline',
        python_model=ensemble_model,
        registered_model_name='Ensemble_Best'
    )

    print("Ensemble registered as Ensemble_Best")

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/07/12 15:15:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 15:15:48 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Registered model 'Ensemble_Best' already exists. Creating a new version of this model...
2026/07/12 15:15:58 INFO mlflow.store.model_registry.abstract_store: Waiting up 

Ensemble registered as Ensemble_Best
🏃 View run Ensemble_Pipeline_Final at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/6/runs/dabacb3b01d64a39ad65d6b47190792e
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/6


In [16]:
test_dates_sorted = sorted(full_data.loc[full_data['Date'] > df_sales_train['Date'].max(), 'Date'].unique())
print(f"Predicting {len(test_dates_sorted)} weeks recursively for both models")

def add_lag_rolling_features(df):
    grp = df.groupby(['Store', 'Dept'])['Weekly_Sales']
    df['lag_1'] = grp.shift(1)
    df['lag_2'] = grp.shift(2)
    df['lag_4'] = grp.shift(4)
    df['lag_52'] = grp.shift(52)

    shifted = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1)
    df['rolling_mean_4'] = shifted.groupby([df['Store'], df['Dept']]).transform(lambda x: x.rolling(4).mean())
    df['rolling_mean_12'] = shifted.groupby([df['Store'], df['Dept']]).transform(lambda x: x.rolling(12).mean())
    df['rolling_std_4'] = shifted.groupby([df['Store'], df['Dept']]).transform(lambda x: x.rolling(4).std())
    df['rolling_mean_52'] = shifted.groupby([df['Store'], df['Dept']]).transform(lambda x: x.rolling(52).mean())
    return df

# We need to track BOTH models' predictions separately, since each model's own
# recursive lag features should be built from ITS OWN prior predictions — otherwise
# one model's errors contaminate the other's lag inputs.
full_data_lgbm = full_data.copy()
full_data_xgb = full_data.copy()

for i, current_date in enumerate(test_dates_sorted):
    full_data_lgbm = add_lag_rolling_features(full_data_lgbm)
    full_data_xgb = add_lag_rolling_features(full_data_xgb)

    mask_today = full_data_lgbm['Date'] == current_date

    X_today_lgbm = full_data_lgbm.loc[mask_today, lgbm_expected_order].copy()
    X_today_lgbm = X_today_lgbm[lgbm_expected_order]  # enforce exact order
    X_today_lgbm = X_today_lgbm.fillna(0)

    X_today_xgb = full_data_xgb.loc[mask_today, xgb_expected_order].copy()
    X_today_xgb['IsHoliday'] = X_today_xgb['IsHoliday'].astype(int)
    X_today_xgb = X_today_xgb[xgb_expected_order]  # enforce exact order
    X_today_xgb = X_today_xgb.fillna(0)

    preds_lgbm_today = lgbm_model.predict(X_today_lgbm)
    preds_xgb_today = xgb_model.predict(X_today_xgb)

    full_data_lgbm.loc[mask_today, 'Weekly_Sales'] = preds_lgbm_today
    full_data_xgb.loc[mask_today, 'Weekly_Sales'] = preds_xgb_today

    if i % 10 == 0:
        print(f"  {current_date.date()} done ({i+1}/{len(test_dates_sorted)})")

print("Recursive prediction complete for both models")

Predicting 39 weeks recursively for both models
  2012-11-02 done (1/39)
  2013-01-11 done (11/39)
  2013-03-22 done (21/39)
  2013-05-31 done (31/39)
Recursive prediction complete for both models


In [17]:
test_lgbm_final = full_data_lgbm[full_data_lgbm['Date'] > df_sales_train['Date'].max()].copy()
test_xgb_final = full_data_xgb[full_data_xgb['Date'] > df_sales_train['Date'].max()].copy()

test_lgbm_final = test_lgbm_final.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
test_xgb_final = test_xgb_final.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

# Sanity check both are aligned on the same rows before combining
assert (test_lgbm_final[['Store','Dept','Date']].values == test_xgb_final[['Store','Dept','Date']].values).all(), \
    "Row alignment mismatch between LightGBM and XGBoost test frames"

final_test_preds = w_lgbm * test_lgbm_final['Weekly_Sales'].values + w_xgb * test_xgb_final['Weekly_Sales'].values
final_test_preds = np.clip(final_test_preds, 0, None)

print(f"Test predictions: {len(final_test_preds)}")
print(f"Min: {final_test_preds.min():.2f}, Max: {final_test_preds.max():.2f}")
print(f"Negative predictions (pre-clip): {(final_test_preds < 0).sum()}")

sample_sub = pd.read_csv('/content/drive/MyDrive/walmart/sampleSubmission.csv')

submission = test_lgbm_final[['Store', 'Dept', 'Date']].copy()
submission['Weekly_Sales'] = final_test_preds
submission['Id'] = (
    submission['Store'].astype(str) + '_' +
    submission['Dept'].astype(str) + '_' +
    submission['Date'].dt.strftime('%Y-%m-%d')
)
submission = submission[['Id', 'Weekly_Sales']]

assert submission.shape[0] == sample_sub.shape[0], f"Row mismatch: {submission.shape[0]} vs {sample_sub.shape[0]}"
assert set(submission['Id']) == set(sample_sub['Id']), "Id mismatch — check date formatting or Store/Dept combos"

submission.to_csv('/content/submission_ensemble.csv', index=False)
print(submission.shape)
submission.head()

Test predictions: 115064
Min: 0.00, Max: 289835.44
Negative predictions (pre-clip): 0
(115064, 2)


,Id,Weekly_Sales
0,1_1_2012-11-02,38837.441165
1,1_1_2012-11-09,22861.440525
2,1_1_2012-11-16,21592.466874
3,1_1_2012-11-23,24054.539759
4,1_1_2012-11-30,28229.596596


In [9]:
!kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f /content/submission_ensemble.csv -m "XGBoost+LightGBM ensemble, optimized weights, recursive inference"
!kaggle competitions submissions -c walmart-recruiting-store-sales-forecasting

100% 3.84M/3.84M [00:00<00:00, 12.9MB/s]
Successfully submitted to Walmart Recruiting - Store Sales ForecastingWarning: Looks like you're using an outdated `kaggle` version (installed: 2.0.2), please consider upgrading to the latest version (2.2.2)
fileName                     date                        description                                                        status                     publicScore  privateScore  
---------------------------  --------------------------  -----------------------------------------------------------------  -------------------------  -----------  ------------  
submission_ensemble.csv      2026-07-12 14:44:36.073000  XGBoost+LightGBM ensemble, optimized weights, recursive inference  SubmissionStatus.PENDING                              
submission.csv               2026-07-11 17:36:25.283000  XGBoost_Best recursive inference                                   SubmissionStatus.COMPLETE  3039.98906   3191.67769    
submission_reduced_lags.csv  2026-0

In [10]:
!kaggle competitions submissions -c walmart-recruiting-store-sales-forecasting

fileName                     date                        description                                                        status                     publicScore  privateScore  
---------------------------  --------------------------  -----------------------------------------------------------------  -------------------------  -----------  ------------  
submission_ensemble.csv      2026-07-12 14:44:36.073000  XGBoost+LightGBM ensemble, optimized weights, recursive inference  SubmissionStatus.COMPLETE  2928.32154   3067.90384    
submission.csv               2026-07-11 17:36:25.283000  XGBoost_Best recursive inference                                   SubmissionStatus.COMPLETE  3039.98906   3191.67769    
submission_reduced_lags.csv  2026-07-11 16:30:00.407000  XGBoost_ReducedLags non-recursive inference                        SubmissionStatus.COMPLETE  12790.49598  13080.77693   
submission.csv               2026-07-11 14:35:01.433000  XGBoost_Best recursive inference                